# Northwind Traders - Relatorio Executivo (Databricks)

Este notebook consolida as 5 visoes analiticas do dashboard para apoio a decisao executiva.

Escopo:
- Pagina 1: Visao Executiva
- Pagina 2: Ticket e Mix de Produtos
- Pagina 3: Clientes e Churn
- Pagina 4: Comercial e Territorio
- Pagina 5: Logistica

Fonte de dados: workspace.indicium_gold no Databricks SQL Warehouse.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime, timedelta
import warnings
import os
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
from databricks import sql
import plotly.io as pio
pio.templates.default = 'plotly_white'

# Carregar .env
env_file = r"c:\Users\Quaresma\Documents\Carreira\Indicium\.env"
load_dotenv(env_file)

print("✅ Todas as bibliotecas carregadas")

✅ Todas as bibliotecas carregadas


In [ ]:
# Conectar ao Databricks
DATABRICKS_HOST = os.getenv("DATABRICKS_HOST")
DATABRICKS_HTTP_PATH = os.getenv("DATABRICKS_HTTP_PATH")
DATABRICKS_TOKEN = os.getenv("DATABRICKS_TOKEN")

conn = sql.connect(
    server_hostname=DATABRICKS_HOST,
    http_path=DATABRICKS_HTTP_PATH,
    personal_access_token=DATABRICKS_TOKEN
)
print("✅ Conectado ao Databricks")

✅ Conectado ao Databricks


In [4]:
# Carregar tabelas do Databricks
def load_table(table_name):
    query = f"SELECT * FROM workspace.indicium_gold.{table_name}"
    return pd.read_sql(query, con=conn)

print("⏳ Carregando dados...")
fact_sales = load_table("fact_sales")
dim_calendar = load_table("dim_calendar")
dim_customer = load_table("dim_customer")
dim_product = load_table("dim_product")   # includes category_name and category_description
dim_employee = load_table("dim_employee")
dim_territory = load_table("dim_territory")  # includes region_description
bridge_employee_territory = load_table("bridge_employee_territory")

# Converter datas principais
for col in ["order_date", "required_date", "shipped_date"]:
    if col in fact_sales.columns:
        fact_sales[col] = pd.to_datetime(fact_sales[col], errors="coerce")

print("✅ Dados carregados com sucesso!")
print(f"   • fact_sales:              {len(fact_sales):,} registros")
print(f"   • dim_product (c/ categ.): {len(dim_product):,} registros")
print(f"   • dim_customer:            {len(dim_customer):,} registros")
print(f"   • dim_employee:            {len(dim_employee):,} registros")
print(f"   • dim_territory:           {len(dim_territory):,} registros")
print(f"   • dim_calendar:            {len(dim_calendar):,} registros")

⏳ Carregando dados...
✅ Dados carregados com sucesso!
   • fact_sales:              2,155 registros
   • dim_product (c/ categ.): 77 registros
   • dim_customer:            91 registros
   • dim_employee:            9 registros
   • dim_territory:           53 registros
   • dim_calendar:            672 registros


---
# PÁGINA 1: VISÃO EXECUTIVA
KPIs principais, receita mensal e por categoria

In [5]:
# KPIs da Visão Executiva
receita_liquida = fact_sales['net_sales'].sum()
receita_bruta = fact_sales['gross_sales'].sum()
total_pedidos = fact_sales['order_id'].nunique()
ticket_medio = receita_liquida / total_pedidos
total_clientes = fact_sales['customer_id'].nunique()

print("\n" + "="*60)
print("📊 VISÃO EXECUTIVA - KPIs")
print("="*60)
print(f"Receita Líquida:    R$ {receita_liquida:>15,.2f}")
print(f"Receita Bruta:      R$ {receita_bruta:>15,.2f}")
print(f"Total de Pedidos:      {total_pedidos:>15,}")
print(f"Ticket Médio:       R$ {ticket_medio:>15,.2f}")
print(f"Clientes Ativos:       {total_clientes:>15,}")
print("="*60)


📊 VISÃO EXECUTIVA - KPIs
Receita Líquida:    R$    1,265,793.04
Receita Bruta:      R$    1,354,458.59
Total de Pedidos:                  830
Ticket Médio:       R$        1,525.05
Clientes Ativos:                    89


In [6]:
# Fig 1: Receita Mensal
receita_mes = fact_sales.groupby(fact_sales['order_date'].dt.to_period('M')).agg({
    'net_sales': 'sum',
    'order_id': 'nunique'
}).reset_index()
receita_mes['data'] = receita_mes['order_date'].dt.to_timestamp()
receita_mes = receita_mes.drop('order_date', axis=1).rename(columns={'net_sales': 'receita', 'order_id': 'pedidos'})

fig = go.Figure()
fig.add_trace(go.Scatter(x=receita_mes['data'], y=receita_mes['receita'], mode='lines+markers', 
                         name='Receita', line=dict(color='#1f77b4', width=3), marker=dict(size=8)))
fig.update_layout(title='📈 Receita Líquida Mensal', xaxis_title='Período', yaxis_title='Receita (R$)', height=400)
fig.show()

In [7]:
# Fig 2: Receita por Categoria  (category_name now inline in dim_product)
fact_cat = fact_sales.merge(dim_product[['product_id', 'category_name']], on='product_id', how='left')
receita_cat = fact_cat.groupby('category_name')['net_sales'].sum().sort_values(ascending=False)

def fmt_cur_short(value):
    if abs(value) >= 1_000_000:
        return f'R$ {value/1_000_000:.1f}M'
    if abs(value) >= 1_000:
        return f'R$ {value/1_000:.1f}K'
    return f'R$ {value:,.0f}'

fig = px.bar(x=receita_cat.values, y=receita_cat.index, orientation='h',
             title='📊 Receita por Categoria', labels={'x': 'Receita (R$)', 'y': 'Categoria'},
             color=receita_cat.values, color_continuous_scale='Viridis',
             text=[fmt_cur_short(v) for v in receita_cat.values])
fig.update_traces(textposition='inside', insidetextanchor='middle', cliponaxis=False)
fig.update_layout(height=400, showlegend=False, uniformtext_minsize=10, uniformtext_mode='hide')
fig.update_yaxes(autorange='reversed')
fig.show()

---
# PÁGINA 2: TICKET E MIX DE PRODUTOS
Análise de desconto, mix e top produtos

In [8]:
# KPIs Página 2
itens_vendidos = fact_sales['quantity'].sum()
desconto_total = fact_sales['discount_value'].sum()
receita_por_cliente = receita_liquida / total_clientes

print("\n" + "="*60)
print("🛍️  TICKET E MIX DE PRODUTOS")
print("="*60)
print(f"Itens Vendidos:          {itens_vendidos:>15,.0f}")
print(f"Desconto Concedido:  R$ {desconto_total:>15,.2f}")
print(f"Receita por Cliente: R$ {receita_por_cliente:>15,.2f}")
print("="*60)


🛍️  TICKET E MIX DE PRODUTOS
Itens Vendidos:                   51,317
Desconto Concedido:  R$       88,665.55
Receita por Cliente: R$       14,222.39


In [9]:
# Fig 3: Desconto vs Ticket (scatter)
pedidos_desc = fact_sales.groupby('order_id').agg({'net_sales': 'sum', 'discount': 'mean'}).reset_index()

fig = px.scatter(pedidos_desc, x='discount', y='net_sales',
                title='💰 Desconto vs Valor do Pedido', 
                labels={'discount': 'Taxa de Desconto', 'net_sales': 'Valor (R$)'},
                opacity=0.6, color='discount', color_continuous_scale='Reds')
fig.update_layout(height=400)
fig.show()

In [10]:
# Fig 4: Top 15 Produtos
fact_prod = fact_sales.merge(dim_product[['product_id', 'product_name']], on='product_id', how='left')
top_prod = fact_prod.groupby('product_name')['net_sales'].sum().sort_values(ascending=False).head(15)

def fmt_cur_short(value):
    if abs(value) >= 1_000_000:
        return f'R$ {value/1_000_000:.1f}M'
    if abs(value) >= 1_000:
        return f'R$ {value/1_000:.1f}K'
    return f'R$ {value:,.0f}'

fig = px.bar(x=top_prod.values, y=top_prod.index, orientation='h',
             title='🏆 Top 15 Produtos por Receita',
             labels={'x': 'Receita (R$)', 'y': 'Produto'},
             color=top_prod.values, color_continuous_scale='Blues',
             text=[fmt_cur_short(v) for v in top_prod.values])
fig.update_traces(textposition='inside', insidetextanchor='middle', cliponaxis=False)
fig.update_layout(height=500, showlegend=False, uniformtext_minsize=9, uniformtext_mode='hide')
fig.update_yaxes(autorange='reversed')
fig.show()

---
# PÁGINA 3: CLIENTES E CHURN
Análise de retenção, inatividade e segmentação

In [11]:
# Análise de Churn e Inatividade
data_ref = fact_sales['order_date'].max()
primeiro_pedido = fact_sales.groupby('customer_id')['order_date'].min()
ultimo_pedido = fact_sales.groupby('customer_id')['order_date'].max()
dias_inativo = (data_ref - ultimo_pedido).dt.days
clientes_risco = (dias_inativo >= 90).sum()
taxa_retencao = ((total_clientes - clientes_risco) / total_clientes * 100)
novos_clientes = (primeiro_pedido == primeiro_pedido.max()).sum()

print("\n" + "="*60)
print("👥 CLIENTES E CHURN")
print("="*60)
print(f"Clientes Ativos:           {total_clientes:>15,}")
print(f"Clientes Novos (últimas):  {novos_clientes:>15,}")
print(f"Clientes em Risco (90+d):  {clientes_risco:>15,}")
print(f"Taxa de Retenção:            {taxa_retencao:>14.2f}%")
print("="*60)


👥 CLIENTES E CHURN
Clientes Ativos:                        89
Clientes Novos (últimas):                1
Clientes em Risco (90+d):               16
Taxa de Retenção:                     82.02%


In [12]:
# Fig 5: Distribuição de Inatividade
dias_df = pd.DataFrame({'dias_inativo': dias_inativo})
fig = px.histogram(dias_df, x='dias_inativo', nbins=40,
                  title='📉 Distribuição de Dias sem Compra',
                  labels={'dias_inativo': 'Dias Inativo', 'count': 'Clientes'},
                  color_discrete_sequence=['#636EFA'])
fig.add_vline(x=90, line_dash='dash', line_color='red', annotation_text='Risco (90d)')
fig.update_layout(height=400)
fig.show()

---
# PÁGINA 4: COMERCIAL E TERRITÓRIO
Ranking de vendedores e análise territorial

In [13]:
# Análise de Vendedores
fact_emp = fact_sales.merge(dim_employee[['employee_id', 'first_name', 'last_name']], on='employee_id', how='left')
fact_emp['vendedor'] = fact_emp['first_name'].fillna('') + ' ' + fact_emp['last_name'].fillna('')
receita_vendedor = fact_emp.groupby('vendedor')['net_sales'].sum().sort_values(ascending=False)
pedidos_vendedor = fact_emp.groupby('vendedor')['order_id'].nunique().sort_values(ascending=False)

print("\n" + "="*60)
print("💼 COMERCIAL E TERRITÓRIO")
print("="*60)
print(f"Top Vendedor: {receita_vendedor.index[0]} (R$ {receita_vendedor.values[0]:,.2f})")
print("\nTop 5 por Receita:")
for i, (v, r) in enumerate(receita_vendedor.head(5).items(), 1):
    print(f"  {i}. {v:40s} R$ {r:>12,.2f}")
print("="*60)


💼 COMERCIAL E TERRITÓRIO
Top Vendedor: Margaret Peacock (R$ 232,890.85)

Top 5 por Receita:
  1. Margaret Peacock                         R$   232,890.85
  2. Janet Leverling                          R$   202,812.84
  3. Nancy Davolio                            R$   192,107.60
  4. Andrew Fuller                            R$   166,537.76
  5. Laura Callahan                           R$   126,862.28


In [14]:
# Fig 6: Ranking Vendedores
def fmt_cur_short(value):
    if abs(value) >= 1_000_000:
        return f'R$ {value/1_000_000:.1f}M'
    if abs(value) >= 1_000:
        return f'R$ {value/1_000:.1f}K'
    return f'R$ {value:,.0f}'

fig = px.bar(x=receita_vendedor.head(10).values, y=receita_vendedor.head(10).index, orientation='h',
             title='🏅 Top 10 Vendedores por Receita',
             labels={'x': 'Receita (R$)', 'y': 'Vendedor'},
             color=receita_vendedor.head(10).values, color_continuous_scale='Plasma',
             text=[fmt_cur_short(v) for v in receita_vendedor.head(10).values])
fig.update_traces(textposition='inside', insidetextanchor='middle', cliponaxis=False)
fig.update_layout(height=400, showlegend=False, uniformtext_minsize=9, uniformtext_mode='hide')
fig.update_yaxes(autorange='reversed')
fig.show()

---
# PÁGINA 5: LOGÍSTICA
Análise de prazos de entrega e impacto

In [15]:
# KPIs Logística
pedidos_prazo = (fact_sales['shipped_on_time_flag'] == 1).sum()
total_pedidos_entrega = len(fact_sales)
pct_prazo = (pedidos_prazo / total_pedidos_entrega * 100)
atraso_medio = fact_sales['shipping_delay_days'].mean()
receita_atraso = fact_sales[fact_sales['shipping_delay_days'] > 0]['net_sales'].sum()

print("\n" + "="*60)
print("📦 LOGÍSTICA")
print("="*60)
print(f"Pedidos no Prazo:         {pedidos_prazo:>15,}")
print(f"% Entregas no Prazo:         {pct_prazo:>14.2f}%")
print(f"Atraso Médio (dias):         {atraso_medio:>14.2f}")
print(f"Receita c/ Atraso:      R$ {receita_atraso:>15,.2f}")
print("="*60)


📦 LOGÍSTICA
Pedidos no Prazo:                   1,990
% Entregas no Prazo:                  92.34%
Atraso Médio (dias):                 -19.47
Receita c/ Atraso:      R$       66,535.61


In [16]:
# Fig 7: Gauge % Prazo
fig = go.Figure(go.Indicator(
    mode='gauge+number+delta',
    value=pct_prazo,
    title={'text': '% Entregas on Top'},
    delta={'reference': 95},
    gauge={
        'axis': {'range': [0, 100]},
        'bar': {'color': '#636EFA'},
        'steps': [
            {'range': [0, 50], 'color': '#f0f0f0'},
            {'range': [50, 90], 'color': '#f5f5f5'}
        ]
    }
))
fig.update_layout(height=400)
fig.show()

In [17]:
# Fig 8: Distribuição de Atrasos
fig = px.histogram(fact_sales, x='shipping_delay_days', nbins=40,
                  title='📊 Distribuição de Atrasos/Adiantamentos',
                  labels={'shipping_delay_days': 'Dias de Atraso (negativo=adiantado)', 'count': 'Pedidos'},
                  color_discrete_sequence=['#AB63FA'])
fig.add_vline(x=0, line_dash='dash', line_color='green', annotation_text='Prazo')
fig.update_layout(height=400)
fig.show()

---
# RESUMO CONSOLIDADO

In [18]:
print("\n" + "#"*65)
print("#" + " "*63 + "#")
print("#  ANÁLISE NORTHWIND - RESUMO EXECUTIVO" + " "*24 + "#")
print("#" + " "*63 + "#")
print("#"*65)

print("\n📊 VISÃO EXECUTIVA")
print(f"   Receita Líquida: R$ {receita_liquida:>15,.2f}")
print(f"   Pedidos: {total_pedidos:>25,}")
print(f"   Ticket Médio: R$ {ticket_medio:>14,.2f}")
print(f"   Clientes: {total_clientes:>26,}")

print("\n🛍️  TICKET E MIX")
print(f"   Itens Vendidos: {itens_vendidos:>20,.0f}")
print(f"   Desconto Total: R$ {desconto_total:>14,.2f}")
print(f"   Receita/Cliente: R$ {receita_por_cliente:>13,.2f}")

print("\n👥 CLIENTES E CHURN")
print(f"   Taxa Retenção: {taxa_retencao:>22.2f}%")
print(f"   Clientes em Risco: {clientes_risco:>15,}")
print(f"   Novos Clientes: {novos_clientes:>19,}")

print("\n💼 COMERCIAL")
print(f"   Top Vendedor: {receita_vendedor.index[0]:>18s}")
print(f"   Vendedores Ativos: {len(receita_vendedor):>14,}")

print("\n📦 LOGÍSTICA")
print(f"   % Entregas no Prazo: {pct_prazo:>17.2f}%")
print(f"   Atraso Médio: {atraso_medio:>24.2f} dias")
print(f"   Receita c/ Atraso: R$ {receita_atraso:>12,.2f}")

print("\n" + "#"*65)
print("\n✅ Análise concluída com sucesso do Databricks!")


#################################################################
#                                                               #
#  ANÁLISE NORTHWIND - RESUMO EXECUTIVO                        #
#                                                               #
#################################################################

📊 VISÃO EXECUTIVA
   Receita Líquida: R$    1,265,793.04
   Pedidos:                       830
   Ticket Médio: R$       1,525.05
   Clientes:                         89

🛍️  TICKET E MIX
   Itens Vendidos:               51,317
   Desconto Total: R$      88,665.55
   Receita/Cliente: R$     14,222.39

👥 CLIENTES E CHURN
   Taxa Retenção:                  82.02%
   Clientes em Risco:              16
   Novos Clientes:                   1

💼 COMERCIAL
   Top Vendedor:   Margaret Peacock
   Vendedores Ativos:              9

📦 LOGÍSTICA
   % Entregas no Prazo:             92.34%
   Atraso Médio:                   -19.47 dias
   Receita c/ Atraso: R$    66,535.61



---
# Exportacao para Entrega

Esta secao gera o arquivo HTML do notebook e tenta exportar em PDF automaticamente.
Se o PDF automatico falhar por dependencia local, use o HTML gerado e faca Impressao > Salvar como PDF no navegador.

In [19]:
%pip install nbconvert -q
print('nbconvert instalado')

Note: you may need to restart the kernel to use updated packages.
nbconvert instalado


In [20]:
%pip install playwright -q
import subprocess, sys
subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=False)
print('playwright/chromium instalados (ou ja existentes)')

Note: you may need to restart the kernel to use updated packages.
playwright/chromium instalados (ou ja existentes)


---
# Apresentacao Executiva (Sem Codigo)

Esta secao gera uma versao de apresentacao para executivos sem exibicao de codigo, apenas com narrativas, graficos e resultados.
Saidas esperadas:
- HTML: analise_northwind_executivo.html
- PDF: analise_northwind_executivo.pdf

In [23]:
import subprocess
import sys
from pathlib import Path
from html import escape

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

notebook_name = "analise_northwind_databricks.ipynb"
candidate_paths = [
    Path.cwd() / notebook_name,
    Path(r"c:\Users\rafae\Documents\Carreira\challenge_indicium") / notebook_name,
]

notebook_file = next((p.resolve() for p in candidate_paths if p.exists()), None)
if notebook_file is None:
    raise FileNotFoundError(f"Notebook não encontrado: {notebook_name}")

dest_dir = notebook_file.parent
exec_pdf = dest_dir / "analise_northwind_executivo.pdf"
assets_dir = dest_dir / "executivo_assets"
assets_dir.mkdir(exist_ok=True)

# HTML intermediário — usado apenas para geração do PDF, apagado ao final.
_tmp_html = dest_dir / "_exec_tmp.html"


def fmt_cur_short(value):
    if abs(value) >= 1_000_000:
        return f'R$ {value/1_000_000:.1f}M'
    if abs(value) >= 1_000:
        return f'R$ {value/1_000:.1f}K'
    return f'R$ {value:,.0f}'


# Fallback: reconstrói o modelo a partir dos CSVs caso o kernel tenha reiniciado.
if "fact_sales" not in globals():
    print("Kernel sem variáveis. Carregando dados locais CSV...")
    data_dir = dest_dir / "data"
    orders = pd.read_csv(data_dir / "orders.csv", sep=";")
    order_details = pd.read_csv(data_dir / "order_details.csv", sep=";")
    products = pd.read_csv(data_dir / "products.csv", sep=";")
    categories = pd.read_csv(data_dir / "categories.csv", sep=";")
    employees = pd.read_csv(data_dir / "employees.csv", sep=";")

    for col in ["order_date", "required_date", "shipped_date"]:
        orders[col] = pd.to_datetime(orders[col], errors="coerce")

    fact_sales = order_details.merge(
        orders[["order_id", "customer_id", "employee_id", "order_date", "required_date", "shipped_date"]],
        on="order_id",
        how="left",
    )
    fact_sales["gross_sales"] = fact_sales["unit_price"] * fact_sales["quantity"]
    fact_sales["net_sales"] = fact_sales["gross_sales"] * (1 - fact_sales["discount"])
    fact_sales["discount_value"] = fact_sales["gross_sales"] - fact_sales["net_sales"]
    fact_sales["shipping_delay_days"] = (fact_sales["shipped_date"] - fact_sales["required_date"]).dt.days
    fact_sales["shipping_delay_days"] = fact_sales["shipping_delay_days"].fillna(0)
    fact_sales["shipped_on_time_flag"] = (fact_sales["shipping_delay_days"] <= 0).astype(int)

    # category_name inline no dim_product (igual ao Gold schema)
    dim_product = products.merge(
        categories[["category_id", "category_name"]], on="category_id", how="left"
    )
    dim_employee = employees[["employee_id", "first_name", "last_name"]].copy()

# KPIs base
receita_liquida = float(fact_sales["net_sales"].sum())
total_pedidos = int(fact_sales["order_id"].nunique())
total_clientes = int(fact_sales["customer_id"].nunique())
ticket_medio = receita_liquida / total_pedidos if total_pedidos else 0.0
pedidos_prazo = int((fact_sales["shipped_on_time_flag"] == 1).sum())
total_pedidos_entrega = int(len(fact_sales))
pct_prazo = (pedidos_prazo / total_pedidos_entrega * 100.0) if total_pedidos_entrega else 0.0
atraso_medio = float(fact_sales["shipping_delay_days"].mean()) if total_pedidos_entrega else 0.0

# Dependência para exportar PNG de gráficos plotly.
try:
    import kaleido  # noqa: F401
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "kaleido", "-q"], check=False)


def save_plot(fig, filename):
    out = assets_dir / filename
    fig.write_image(str(out), format="png", width=1400, height=800, scale=2)
    return out.name


# ── Dados para os gráficos ────────────────────────────────────────────────────
receita_mes = fact_sales.groupby(fact_sales["order_date"].dt.to_period("M")).agg({
    "net_sales": "sum",
    "order_id": "nunique",
}).reset_index()
receita_mes["data"] = receita_mes["order_date"].dt.to_timestamp()
receita_mes = receita_mes.drop("order_date", axis=1).rename(columns={
    "net_sales": "receita",
    "order_id": "pedidos",
})

# category_name já está inline no dim_product (Gold schema)
fact_cat = fact_sales.merge(dim_product[["product_id", "category_name"]], on="product_id", how="left")
receita_cat = fact_cat.groupby("category_name")["net_sales"].sum().sort_values(ascending=False)

pedidos_desc = fact_sales.groupby("order_id").agg({"net_sales": "sum", "discount": "mean"}).reset_index()

fact_prod = fact_sales.merge(dim_product[["product_id", "product_name"]], on="product_id", how="left")
top_prod = fact_prod.groupby("product_name")["net_sales"].sum().sort_values(ascending=False).head(15)

data_ref = fact_sales["order_date"].max()
ultimo_pedido = fact_sales.groupby("customer_id")["order_date"].max()
dias_inativo = (data_ref - ultimo_pedido).dt.days
dias_df = pd.DataFrame({"dias_inativo": dias_inativo})

fact_emp = fact_sales.merge(dim_employee[["employee_id", "first_name", "last_name"]], on="employee_id", how="left")
fact_emp["vendedor"] = fact_emp["first_name"].fillna("") + " " + fact_emp["last_name"].fillna("")
receita_vendedor = fact_emp.groupby("vendedor")["net_sales"].sum().sort_values(ascending=False)

# ── Métricas derivadas ────────────────────────────────────────────────────────
top_categoria = receita_cat.index[0]
top_categoria_valor = float(receita_cat.iloc[0])
top_categoria_pct = (top_categoria_valor / receita_liquida * 100.0) if receita_liquida else 0.0
media_desconto = float(pedidos_desc["discount"].mean())
media_ticket_pedido = float(pedidos_desc["net_sales"].mean())
top_produto = top_prod.index[0]
top_produto_valor = float(top_prod.iloc[0])
top_vendedor = receita_vendedor.index[0]
top_vendedor_valor = float(receita_vendedor.iloc[0])
clientes_em_risco_pct = (int((dias_inativo > 90).sum()) / total_clientes * 100.0) if total_clientes else 0.0
atraso_mediano = float(dias_df["dias_inativo"].median())
atraso_medio_neg = float(fact_sales.loc[fact_sales["shipping_delay_days"] < 0, "shipping_delay_days"].mean()) if (fact_sales["shipping_delay_days"] < 0).any() else 0.0
atraso_medio_pos = float(fact_sales.loc[fact_sales["shipping_delay_days"] > 0, "shipping_delay_days"].mean()) if (fact_sales["shipping_delay_days"] > 0).any() else 0.0

# ── Gráficos ──────────────────────────────────────────────────────────────────
fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=receita_mes["data"],
    y=receita_mes["receita"],
    mode="lines+markers",
    line=dict(color="#0B6E4F", width=3),
    marker=dict(size=7),
    name="Receita",
))
fig1.update_layout(title="Receita Líquida Mensal", xaxis_title="Período", yaxis_title="Receita (R$)")
if not receita_mes.empty:
    fig1.add_annotation(
        x=receita_mes["data"].iloc[-1],
        y=receita_mes["receita"].iloc[-1],
        text=f"Último mês: {fmt_cur_short(receita_mes['receita'].iloc[-1])}",
        showarrow=True, arrowhead=2, ax=40, ay=-30, bgcolor="white",
    )

fig2 = px.bar(
    x=receita_cat.values, y=receita_cat.index, orientation="h",
    title="Receita por Categoria", labels={"x": "Receita (R$)", "y": "Categoria"},
    color=receita_cat.values, color_continuous_scale="Tealgrn",
    text=[fmt_cur_short(v) for v in receita_cat.values],
)
fig2.update_traces(textposition="inside", insidetextanchor="middle", cliponaxis=False)
fig2.update_yaxes(autorange="reversed")

fig3 = px.scatter(
    pedidos_desc, x="discount", y="net_sales",
    title="Desconto vs. Valor do Pedido",
    labels={"discount": "Taxa de Desconto", "net_sales": "Valor (R$)"},
    opacity=0.65, color="discount", color_continuous_scale="Reds",
)

fig4 = px.bar(
    x=top_prod.values, y=top_prod.index, orientation="h",
    title="Top 15 Produtos por Receita", labels={"x": "Receita (R$)", "y": "Produto"},
    color=top_prod.values, color_continuous_scale="Blues",
    text=[fmt_cur_short(v) for v in top_prod.values],
)
fig4.update_traces(textposition="inside", insidetextanchor="middle", cliponaxis=False)
fig4.update_yaxes(autorange="reversed")

fig5 = px.histogram(
    dias_df, x="dias_inativo", nbins=40,
    title="Distribuição de Dias sem Compra",
    labels={"dias_inativo": "Dias Inativo", "count": "Clientes"},
    color_discrete_sequence=["#0B6E4F"],
)
fig5.add_vline(x=90, line_dash="dash", line_color="red")

fig6 = px.bar(
    x=receita_vendedor.head(10).values, y=receita_vendedor.head(10).index, orientation="h",
    title="Top 10 Vendedores por Receita", labels={"x": "Receita (R$)", "y": "Vendedor"},
    color=receita_vendedor.head(10).values, color_continuous_scale="Plasma",
    text=[fmt_cur_short(v) for v in receita_vendedor.head(10).values],
)
fig6.update_traces(textposition="inside", insidetextanchor="middle", cliponaxis=False)
fig6.update_yaxes(autorange="reversed")

fig7 = go.Figure(go.Indicator(
    mode="gauge+number+delta",
    value=float(pct_prazo),
    title={"text": "% Entregas no Prazo"},
    delta={"reference": 95},
    gauge={
        "axis": {"range": [0, 100]},
        "bar": {"color": "#0B6E4F"},
        "steps": [
            {"range": [0, 50], "color": "#ECECEC"},
            {"range": [50, 90], "color": "#DCEFE8"},
        ],
    },
))

fig8 = px.histogram(
    fact_sales, x="shipping_delay_days", nbins=40,
    title="Distribuição de Atrasos e Adiantamentos",
    labels={"shipping_delay_days": "Dias de Atraso", "count": "Pedidos"},
    color_discrete_sequence=["#1F77B4"],
)
fig8.add_vline(x=0, line_dash="dash", line_color="green")

# ── Exportar PNGs e montar HTML ───────────────────────────────────────────────
print("Gerando gráficos estáticos (PNG)...")
charts_data = [
    (save_plot(fig1, "01_receita_mensal.png"),    "Receita Líquida Mensal",          f"A última leitura do período é de {fmt_cur_short(receita_mes['receita'].iloc[-1])}."),
    (save_plot(fig2, "02_receita_categoria.png"), "Receita por Categoria",            f"{top_categoria} lidera com {fmt_cur_short(top_categoria_valor)} ({top_categoria_pct:.1f}% do total)."),
    (save_plot(fig3, "03_desconto_vs_ticket.png"),"Desconto vs. Valor do Pedido",     f"Desconto médio: {media_desconto:.1%}. Ticket médio por pedido: {fmt_cur_short(media_ticket_pedido)}."),
    (save_plot(fig4, "04_top_produtos.png"),      "Top 15 Produtos por Receita",      f"Produto líder: {top_produto} — {fmt_cur_short(top_produto_valor)}."),
    (save_plot(fig5, "05_inatividade_clientes.png"),"Distribuição de Inatividade",    f"Mediana de inatividade: {atraso_mediano:.0f} dias. {clientes_em_risco_pct:.1f}% da base em risco (>90d)."),
    (save_plot(fig6, "06_top_vendedores.png"),    "Top 10 Vendedores por Receita",    f"{top_vendedor} lidera com {fmt_cur_short(top_vendedor_valor)}."),
    (save_plot(fig7, "07_gauge_prazo.png"),       "% Entregas no Prazo",              f"Nível atual: {pct_prazo:.2f}% (meta: 95%)."),
    (save_plot(fig8, "08_atrasos.png"),           "Distribuição de Atrasos",          f"Adiantamento médio: {atraso_medio_neg:.1f}d. Atraso médio: {atraso_medio_pos:.1f}d."),
]

kpi_cards = [
    ("Receita Líquida",      f"R$ {receita_liquida:,.2f}"),
    ("Total de Pedidos",     f"{total_pedidos:,}"),
    ("Ticket Médio",         f"R$ {ticket_medio:,.2f}"),
    ("Clientes Ativos",      f"{total_clientes:,}"),
    ("% Entregas no Prazo",  f"{pct_prazo:.2f}%"),
    ("Atraso Médio",         f"{atraso_medio:.2f} dias"),
]

cards_html = "\n".join([
    f"<div class='card'><h3>{escape(k)}</h3><p>{escape(v)}</p></div>"
    for k, v in kpi_cards
])

charts_html = "\n".join([
    f"<section class='chart'>"
    f"<h2>{escape(title)}</h2>"
    f"<p class='chart-text'>{escape(desc)}</p>"
    f"<img src='executivo_assets/{escape(img)}' alt='{escape(title)}'>"
    f"</section>"
    for img, title, desc in charts_data
])

html_content = f"""<!doctype html>
<html lang='pt-BR'>
<head>
  <meta charset='utf-8'>
  <title>Northwind Traders - Apresentação Executiva</title>
  <style>
    @page {{ size: A4 portrait; margin: 12mm; }}
    body {{ font-family: Segoe UI, Calibri, Arial, sans-serif; margin: 0; color: #1B1F24; background: #F7FAF9; }}
    .wrap {{ max-width: 1080px; margin: 0 auto; padding: 16px 18px 24px; }}
    h1 {{ margin: 0 0 8px; font-size: 30px; color: #0B6E4F; }}
    .sub {{ margin: 0 0 20px; color: #4B5563; font-size: 16px; line-height: 1.4; }}
    .cards {{ display: grid; grid-template-columns: repeat(3, 1fr); gap: 12px; margin-bottom: 18px; }}
    .card {{ background: #fff; border: 1px solid #D7E5DE; border-radius: 12px; padding: 12px; }}
    .card h3 {{ margin: 0 0 6px; font-size: 13px; color: #4B5563; font-weight: 600; }}
    .card p {{ margin: 0; font-size: 23px; color: #0F172A; font-weight: 700; }}
    .chart {{ background: #fff; border: 1px solid #D7E5DE; border-radius: 12px; padding: 12px; margin: 0 0 14px; page-break-inside: avoid; }}
    .chart h2 {{ margin: 0 0 6px; font-size: 18px; color: #0F172A; }}
    .chart-text {{ margin: 0 0 10px; font-size: 14px; color: #4B5563; line-height: 1.45; }}
    .chart img {{ width: 100%; height: auto; display: block; }}
    .footer {{ margin-top: 8px; font-size: 12px; color: #6B7280; }}
  </style>
</head>
<body>
  <div class='wrap'>
    <h1>Northwind Traders — Apresentação Executiva</h1>
    <p class='sub'>Painel consolidado para decisão executiva: performance comercial, clientes e logística.</p>
    <div class='cards'>{cards_html}</div>
    {charts_html}
    <p class='footer'>Relatório gerado automaticamente a partir do notebook analítico.</p>
  </div>
</body>
</html>"""

_tmp_html.write_text(html_content, encoding="utf-8")

# ── Gerar PDF via Playwright ──────────────────────────────────────────────────
print("Gerando PDF executivo...")
pdf_script = f'''
from pathlib import Path
from playwright.sync_api import sync_playwright

html_file = Path(r"{_tmp_html}").resolve()
pdf_file  = Path(r"{exec_pdf}").resolve()

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page(viewport={{"width": 1400, "height": 2200}})
    page.goto(html_file.as_uri(), wait_until="networkidle")
    page.wait_for_selector("img", timeout=30000)
    page.pdf(
        path=str(pdf_file),
        format="A4",
        print_background=True,
        margin={{"top": "10mm", "right": "10mm", "bottom": "10mm", "left": "10mm"}}
    )
    browser.close()

print("OK: PDF executivo gerado em " + str(pdf_file))
'''

run_pdf = subprocess.run([sys.executable, "-c", pdf_script], capture_output=True, text=True)

# Remove o HTML intermediário independentemente do resultado.
_tmp_html.unlink(missing_ok=True)

if run_pdf.returncode == 0 and exec_pdf.exists():
    print(run_pdf.stdout.strip())
    print(f"Gráficos exportados: {len(charts_data)}")
else:
    print("Falha ao gerar PDF executivo.")
    print(run_pdf.stderr[:2000])

Gerando gráficos estáticos (PNG)...
Gerando PDF executivo...
OK: PDF executivo gerado em C:\Users\rafae\Documents\Carreira\challenge_indicium\analise_northwind_executivo.pdf
Gráficos exportados: 8
